[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/main/examples/function_calling.ipynb)

# Call an ordinary function

The assistant can check the weather, change a tariff, look up an order or read a balance. The arguments that come from a closed set are questions. The arguments that are numbers are a regular expression. An LLM would emit a JSON tool call you then validate. FRIDA-Decisions picks the function and each closed argument in one request; code calls a normal Python function, or asks a follow-up when `confidence` is low.

Every argument question is asked even when the function does not need it. Unused answers are ignored. Extra questions are cheap next to one copy of the text, and the code decides which of them matter.


In [ ]:
%pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [1]:
import os
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


frida-decisions 0.4.0, loaded on cpu, torch.float32


One request asks for the function and every closed argument. Code ignores the arguments the chosen function does not use. `0.55` is the confidence below which the code asks again instead of calling.


In [2]:
import re

DATA = [
    {"text": "Какая завтра погода в Казани?", "outcome": "weather:kzn"},
    {"text": "Переведите меня на безлимитный тариф на год.", "outcome": "tariff:unlimited:year"},
    {"text": "Где мой заказ №5512?", "outcome": "order:5512"},
    {"text": "Сколько денег на карте?", "outcome": "balance"},
    {"text": "Хочу сменить тариф.", "outcome": "clarify"},
    {"text": "Погода.", "outcome": "clarify"},
    {"text": "Проверьте статус заказа 8841.", "outcome": "order:8841"},
    {"text": "Баланс счёта, пожалуйста, кредит не предлагайте.", "outcome": "balance"},
]
FUNCTIONS = {
    "weather": "прогноз погоды в названном городе",
    "tariff": "сменить тариф мобильной связи",
    "order": "узнать статус заказа по его номеру",
    "balance": "узнать остаток денег на счёте или карте",
}
CITY = {
    "msk": "Москва", "spb": "Санкт-Петербург", "kzn": "Казань", "sochi": "Сочи",
    "none": "город не назван",
}
PLAN = {
    "basic": "базовый тариф, мало интернета",
    "smart": "средний тариф с пакетом минут и гигабайт",
    "unlimited": "безлимитный интернет",
    "none": "тариф не назван",
}
PERIOD = {
    "month": "на один месяц", "year": "сразу на год", "none": "срок не назван",
}
MIN_CONFIDENCE = 0.55
ORDER_RE = re.compile(r"(?:заказ\w*|№)\s*№?\s*(\d{3,})", re.IGNORECASE)


def request_for(text):
    return {
        "state": text,
        "questions": {
            "fn": {"type": "choice", "instructions": "Какую функцию вызвать?", "criteria": FUNCTIONS},
            "city": {"type": "choice", "instructions": "О каком городе речь? Если город не назван, выбери none.",
                     "criteria": CITY},
            "plan": {"type": "choice", "instructions": "Какой тариф назван? Если не назван, выбери none.",
                     "criteria": PLAN},
            "period": {"type": "choice", "instructions": "На какой срок менять тариф? Если срок не назван, выбери none.",
                       "criteria": PERIOD},
        },
    }


def weather(city):
    return f"Завтра в городе {CITY[city]} без осадков."


def change_tariff(plan, period):
    return f"Заявка: тариф {PLAN[plan]}, срок {PERIOD[period]}."


def order_status(number):
    return f"Заказ {number}: передан в доставку."


def balance():
    return "Остаток на счёте: 1 240 рублей."


def decide(text, answers):
    fn = answers["fn"]
    if fn["confidence"] < MIN_CONFIDENCE:
        return {"outcome": "clarify", "detail": "Уточните: погода, тариф, статус заказа или баланс.",
                "confidence": fn["confidence"]}
    name = fn["choice"]
    if name == "weather":
        city = answers["city"]["choice"]
        if city == "none":
            return {"outcome": "clarify", "detail": "В каком городе посмотреть погоду?",
                    "confidence": fn["confidence"]}
        return {"outcome": f"weather:{city}", "detail": weather(city), "confidence": fn["confidence"]}
    if name == "tariff":
        plan = answers["plan"]["choice"]
        period = answers["period"]["choice"]
        if plan == "none" or period == "none":
            return {"outcome": "clarify", "detail": "Какой тариф и на какой срок?",
                    "confidence": fn["confidence"]}
        return {"outcome": f"tariff:{plan}:{period}", "detail": change_tariff(plan, period),
                "confidence": fn["confidence"]}
    if name == "order":
        found = ORDER_RE.search(text)
        if not found:
            return {"outcome": "clarify", "detail": "Назовите номер заказа.",
                    "confidence": fn["confidence"]}
        return {"outcome": f"order:{found.group(1)}", "detail": order_status(found.group(1)),
                "confidence": fn["confidence"]}
    return {"outcome": "balance", "detail": balance(), "confidence": fn["confidence"]}


`outcome` is what the code did: a concrete call, or `clarify`. The gold column has the same shape.


In [3]:
def run(judge):
    hits = 0
    for row in DATA:
        decision = decide(row["text"], judge(request_for(row["text"]))["answers"])
        if decision["outcome"] == row["outcome"]:
            mark = "ok"
            hits += 1
        else:
            mark = "miss"
        print(f"{mark:4} got {decision['outcome']:22} gold {row['outcome']:22} "
              f"p={decision['confidence']:.2f}  {decision['detail']}")
    print(f"\n{hits}/{len(DATA)} outcomes match the labels")


In [4]:
run(judge)


ok   got weather:kzn            gold weather:kzn            p=1.00  Завтра в городе Казань без осадков.
ok   got tariff:unlimited:year  gold tariff:unlimited:year  p=1.00  Заявка: тариф безлимитный интернет, срок сразу на год.
ok   got order:5512             gold order:5512             p=1.00  Заказ 5512: передан в доставку.
ok   got balance                gold balance                p=1.00  Остаток на счёте: 1 240 рублей.
ok   got clarify                gold clarify                p=1.00  Какой тариф и на какой срок?
ok   got clarify                gold clarify                p=1.00  В каком городе посмотреть погоду?
ok   got order:8841             gold order:8841             p=1.00  Заказ 8841: передан в доставку.
ok   got balance                gold balance                p=1.00  Остаток на счёте: 1 240 рублей.

8/8 outcomes match the labels
